<a href="https://colab.research.google.com/github/sjsu-cs131-spring26/trendtrackers-engagement-socialmedia/blob/pa5-joshua/notebooks/trendtrackers-pa5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Sprint 5 - Trend Trackers**


In [ ]:
!pip install pyspark


In [ ]:
import pyspark.sql.functions as sf
from pyspark.sql import SparkSession
from pyspark.sql.window import Window


In [ ]:
spark = (SparkSession.builder
    .appName("trendtrackers-pa5")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate())

print("Spark version:", spark.version)
print("Shuffle partitions:", spark.conf.get("spark.sql.shuffle.partitions"))


Spark version: 4.0.2
Shuffle partitions: 8


## 1) Data Loading
Load the dataset into a PySpark DataFrame and verify the schema.


In [ ]:
!wget -O sample10k.tsv https://raw.githubusercontent.com/sjsu-cs131-spring26/trendtrackers-engagement-socialmedia/main/data/sample10k.tsv


--2026-04-19 19:00:52--  https://raw.githubusercontent.com/sjsu-cs131-spring26/trendtrackers-engagement-socialmedia/main/data/sample10k.tsv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 3875902 (3.7M) [text/plain]
Saving to: ‘sample10k.tsv’

sample10k.tsv       100%[===================>]   3.70M  --.-KB/s    in 0.05s   

2026-04-19 19:00:53 (75.7 MB/s) - ‘sample10k.tsv’ saved [3875902/3875902]



In [ ]:
df = (spark.read
    .option("header", True)
    .option("inferSchema", False)
    .option("sep", "\t")
    .option("multiLine", True)
    .option("mode", "PERMISSIVE")
    .csv("sample10k.tsv"))

df.printSchema()
df.show(5, truncate=False)

print("Row count:", df.count())
print("Columns:", df.columns)

root
 |-- date: string (nullable = true)
 |-- original_text: string (nullable = true)
 |-- url: string (nullable = true)
 |-- author_hash: string (nullable = true)
 |-- language: string (nullable = true)
 |-- primary_theme: string (nullable = true)
 |-- english_keywords: string (nullable = true)
 |-- sentiment: string (nullable = true)
 |-- main_emotion: string (nullable = true)
 |-- secondary_themes: string (nullable = true)

+--------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----------------------------------------------------------------------------------------------------------------------+-------------------------------------------------------------------------------------------------------------+----------------------------------------+--------+---------------------------------------------------------------------------------------------------------------------------

## 2) Data Cleaning / Quality Filtering
Remove nulls, remove invalid rows, normalize important columns, and safely handle bad dates.


In [ ]:
clean_df = (df
    .withColumn("primary_theme", sf.trim(sf.lower(sf.col("primary_theme"))))
    .withColumn("sentiment", sf.trim(sf.col("sentiment").cast("string")))
    .withColumn("date", sf.trim(sf.col("date").cast("string"))))

clean_df = clean_df.filter(
    sf.col("primary_theme").isNotNull() &
    sf.col("date").isNotNull() &
    sf.col("sentiment").isNotNull() &
    (sf.col("primary_theme") != "") &
    (sf.col("date") != "") &
    (sf.col("sentiment") != "")
)

clean_df = clean_df.filter(
    sf.col("sentiment").rlike(r"^-?\d+(\.\d+)?$")
)

clean_df = clean_df.withColumn("sentiment", sf.col("sentiment").cast("double"))

clean_df = clean_df.withColumn(
    "date_ts",
    sf.try_to_timestamp(
        sf.col("date"),
        sf.lit("yyyy-MM-dd'T'HH:mm:ss.SSS'Z'")
    )
)

clean_df = clean_df.filter(sf.col("date_ts").isNotNull())

print("Original rows:", df.count())
print("Clean rows:", clean_df.count())
clean_df.select("date", "date_ts", "primary_theme", "sentiment").show(10, truncate=False)


Original rows: 10214
Clean rows: 9884
+------------------------+-------------------+-------------+---------+
|date                    |date_ts            |primary_theme|sentiment|
+------------------------+-------------------+-------------+---------+
|2024-12-01T00:38:21.000Z|2024-12-01 00:38:21|entertainment|-0.47    |
|2024-12-01T00:27:20.000Z|2024-12-01 00:27:20|entertainment|0.05     |
|2024-12-01T00:38:16.000Z|2024-12-01 00:38:16|entertainment|-0.75    |
|2024-12-01T00:54:25.000Z|2024-12-01 00:54:25|people       |-0.72    |
|2024-12-01T00:41:55.000Z|2024-12-01 00:41:55|politics     |-0.2     |
|2024-12-01T01:03:04.000Z|2024-12-01 01:03:04|social       |-0.56    |
|2024-12-01T00:06:03.000Z|2024-12-01 00:06:03|sports       |0.14     |
|2024-12-01T00:30:06.000Z|2024-12-01 00:30:06|politics     |-0.58    |
|2024-12-01T00:10:56.000Z|2024-12-01 00:10:56|people       |-0.68    |
|2024-12-01T00:18:34.000Z|2024-12-01 00:18:34|law          |0.13     |
+------------------------+-------------

## 3) PySpark Transform / Frequency Table
Create a frequency table with `groupBy()` and aggregation.


In [ ]:
theme_frequency = (clean_df
    .groupBy("primary_theme")
    .agg(sf.count("*").alias("theme_count"))
    .orderBy(sf.col("theme_count").desc()))

theme_frequency.show(20, truncate=False)


+--------------+-----------+
|primary_theme |theme_count|
+--------------+-----------+
|entertainment |2093       |
|people        |1565       |
|sports        |1163       |
|politics      |1063       |
|technology    |906        |
|environment   |562        |
|social        |479        |
|health        |440        |
|science       |389        |
|cryptocurrency|379        |
|business      |241        |
|finance       |177        |
|economy       |160        |
|investing     |152        |
|law           |115        |
+--------------+-----------+



## 4) PySpark Transform / Top-N Analysis
Generate the Top 10 ranked entities using aggregation and sorting.


In [ ]:
top_10_themes = (clean_df
    .groupBy("primary_theme")
    .agg(sf.count("*").alias("theme_count"))
    .orderBy(sf.col("theme_count").desc())
    .limit(10))

top_10_themes.show(truncate=False)


+--------------+-----------+
|primary_theme |theme_count|
+--------------+-----------+
|entertainment |2093       |
|people        |1565       |
|sports        |1163       |
|politics      |1063       |
|technology    |906        |
|environment   |562        |
|social        |479        |
|health        |440        |
|science       |389        |
|cryptocurrency|379        |
+--------------+-----------+



## 5) Advanced Transform / Window Function
Use a window function to rank rows inside each theme by sentiment.


In [ ]:
window_spec = Window.partitionBy("primary_theme").orderBy(sf.col("sentiment").desc())

ranked_df = clean_df.withColumn("rank_in_theme", sf.row_number().over(window_spec))

ranked_df.select("primary_theme", "sentiment", "rank_in_theme", "date") \
    .orderBy("primary_theme", "rank_in_theme") \
    .show(30, truncate=False)


+-------------+---------+-------------+------------------------+
|primary_theme|sentiment|rank_in_theme|date                    |
+-------------+---------+-------------+------------------------+
|business     |0.86     |1            |2024-12-01T00:49:56.000Z|
|business     |0.82     |2            |2024-12-01T00:41:54.000Z|
|business     |0.82     |3            |2024-12-01T00:30:13.000Z|
|business     |0.8      |4            |2024-12-01T00:05:47.000Z|
|business     |0.78     |5            |2024-12-01T00:00:02.000Z|
|business     |0.76     |6            |2024-12-01T00:51:13.000Z|
|business     |0.76     |7            |2024-12-01T00:59:40.000Z|
|business     |0.76     |8            |2024-12-01T00:17:13.000Z|
|business     |0.74     |9            |2024-12-01T00:16:25.000Z|
|business     |0.73     |10           |2024-12-01T00:08:30.000Z|
|business     |0.72     |11           |2024-12-01T00:16:45.000Z|
|business     |0.71     |12           |2024-12-01T00:34:28.000Z|
|business     |0.71     |

In [ ]:
top_3_per_theme = ranked_df.filter(sf.col("rank_in_theme") <= 3)

top_3_per_theme.orderBy("primary_theme", "rank_in_theme") \
    .show(50, truncate=False)


+------------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----------------------------------------------------------------------------------------------------+----------------------------------------+--------+--------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

## 6) Trend Analysis / Time Aggregation
Create a month-level trend table from the timestamp column.


In [ ]:
monthly_trend = (clean_df
    .withColumn("month", sf.date_format(sf.col("date_ts"), "yyyy-MM"))
    .groupBy("month")
    .agg(
        sf.count("*").alias("row_count"),
        sf.avg("sentiment").alias("avg_sentiment")
    )
    .orderBy("month"))

monthly_trend.show(truncate=False)


+-------+---------+-------------------+
|month  |row_count|avg_sentiment      |
+-------+---------+-------------------+
|2024-12|9884     |0.04167341157426146|
+-------+---------+-------------------+



## 7) Data Cleaning
Removing nulls, invalid rows, normalize columns

In [ ]:
#normalize important columns
clean_df = (df
    .withColumn("date", sf.trim(sf.col("date").cast("string")))
    .withColumn("original_text", sf.trim(sf.col("original_text").cast("string")))
    .withColumn("language", sf.lower(sf.trim(sf.col("language").cast("string"))))
    .withColumn("primary_theme", sf.lower(sf.trim(sf.col("primary_theme").cast("string"))))
    .withColumn("main_emotion", sf.lower(sf.trim(sf.col("main_emotion").cast("string"))))
    .withColumn("sentiment", sf.trim(sf.col("sentiment").cast("string")))
)

In [ ]:
#remove rows with null or empty values in required columns

clean_df = clean_df.filter(
    sf.col("date").isNotNull() &
    sf.col("original_text").isNotNull() &
    sf.col("language").isNotNull() &
    sf.col("primary_theme").isNotNull() &
    sf.col("sentiment").isNotNull() &
    (sf.col("date") != "") &
    (sf.col("original_text") != "") &
    (sf.col("language") != "") &
    (sf.col("primary_theme") != "") &
    (sf.col("sentiment") != "")
)


In [ ]:
#keep only valid numeric sentiment values

clean_df = clean_df.filter(
    sf.col("sentiment").rlike(r"^-?\d+(\.\d+)?$")
)

In [ ]:
#cast sentiments to double

clean_df = clean_df.withColumn("sentiment", sf.col("sentiment").cast("double"))

In [ ]:
#keep only valid numeric sentiments values

clean_df = clean_df.filter(
    sf.col("sentiment").rlike(r"^-?\d+(\.\d+)?$")
)

In [ ]:
#cast sentiments to double
clean_df = clean_df.withColumn("sentiment", sf.col("sentiment").cast("double"))

In [ ]:
#keep only valid sentiments range [-1, 1]

clean_df = clean_df.filter(
    (sf.col("sentiment") >= -1.0) &
    (sf.col("sentiment") <= 1.0)
)


In [ ]:
#safely parse timelapse

clean_df = clean_df.withColumn(
    "date_ts",
    sf.try_to_timestamp(
        sf.col("date"),
        sf.lit("yyyy-MM-dd'T'HH:mm:ss.SSS'Z'")
    )
)

In [ ]:
#remove bad timestamps

clean_df = clean_df.filter(sf.col("date_ts").isNotNull())

In [ ]:
#remove duplicate posts

clean_df = clean_df.dropDuplicates(["original_text", "url"])

## 8) Verifying Cleaned Data

In [ ]:
print("Original rows:", df.count())
print("Clean rows:", clean_df.count())

print("=== CLEANED SCHEMA ===")
clean_df.printSchema()

print("=== CLEANED SAMPLE ===")
clean_df.select(
    "date", "date_ts", "original_text", "language",
    "primary_theme", "main_emotion", "sentiment"
).show(10, truncate=False)

# 9) Creating A Clean Base Table


In [ ]:
clean_posts = clean_df.select(
    "date_ts",
    "date",
    "original_text",
    "url",
    "author_hash",
    "language",
    "primary_theme",
    "english_keywords",
    "sentiment",
    "main_emotion",
    "secondary_themes"
)

print("=== CLEAN POSTS TABLE ===")
clean_posts.show(10, truncate=False)

# 10) Showing Theme Frequency


In [ ]:
theme_frequency = (clean_posts
    .groupBy("primary_theme")
    .agg(sf.count("*").alias("theme_count"))
    .orderBy(sf.col("theme_count").desc()))

print("=== THEME FREQUENCY ===")
theme_frequency.show(20, truncate=False)

# 11) Montlhy Sentiments Summary

In [ ]:
monthly_summary = (clean_posts
    .withColumn("month", sf.date_format(sf.col("date_ts"), "yyyy-MM"))
    .groupBy("month")
    .agg(
        sf.count("*").alias("post_count"),
        sf.avg("sentiment").alias("avg_sentiment")
    )
    .orderBy("month"))

print("=== MONTHLY SUMMARY ===")
monthly_summary.show(truncate=False)

# 12) Saving output task


In [ ]:
lean_posts.write.mode("overwrite").parquet("clean_posts.parquet")
theme_frequency.write.mode("overwrite").parquet("theme_frequency.parquet")
monthly_summary.write.mode("overwrite").parquet("monthly_summary.parquet")

print("Saved:")
print("- clean_posts.parquet")
print("- theme_frequency.parquet")
print("- monthly_summary.parquet")